# Tools in Langchain - 2 

# 1. Short Term Memory

In [6]:
from langchain_mistralai import ChatMistralAI 
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage
from langchain.agents import create_agent
from dotenv import load_dotenv 
load_dotenv()

True

In [5]:
@tool 
def get_last_user_message(runtime:ToolRuntime) -> str:
    """
    This tool is used to access the last user message if it exist.
    """
    messages = runtime.state['messages'] 

    for message in reversed(messages):
        if isinstance(message,HumanMessage):
            return message.content
    return "No user messages found."

@tool 
def get_user_preference(pref : str,runtime:ToolRuntime) -> str:
    """
    This tool is used to access the user preference if it exist.
    """
    preferences = runtime.state.get("User_preference",{"Sarcastic"})
    return preferences.get(pref,"Not set")

In [7]:
model = ChatMistralAI(
    model = "mistral-medium-2508"
)

In [8]:
agent = create_agent(
    model = model,
    tools = [get_last_user_message,get_user_preference]
)

In [9]:
response = agent.invoke(
    {
        "messages" : [("user","hello how are you")]
    }
)

In [10]:
response

{'messages': [HumanMessage(content='hello how are you', additional_kwargs={}, response_metadata={}, id='88c24392-71be-4ce0-9420-3a9c003dd2fb'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'FeYpaQscM', 'function': {'name': 'get_last_user_message', 'arguments': '{}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 126, 'total_tokens': 134, 'completion_tokens': 8, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-2508', 'model': 'mistral-medium-2508', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cdbc7-b8e2-79c0-8bf9-ecc05bcf96b2-0', tool_calls=[{'name': 'get_last_user_message', 'args': {}, 'id': 'FeYpaQscM', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 126, 'output_tokens': 8, 'total_tokens': 134}),
  ToolMessage(content='hello how are you', name='get_last_user_message', id='10a74f4c-7552-4503-a8dc-3221f266d553', tool_call_id='FeYpaQscM'),
  AIMessage(cont

# Context based tools

In [12]:
from dataclasses import dataclass
from langchain_mistralai import ChatMistralAI 
from langchain.agents import create_agent

In [13]:
Company_db = {

    "user1910":{
        "name" : "Dhairya",
        "department" : "AI"
    },
    "user1019":{
        "name" : "Simple",
        "department" : "Web"

    }
}

In [21]:
@dataclass
class UserContext():
    user_id : str 

@tool 
def get_user_info(runtime : ToolRuntime[UserContext]):
    """
    returns user information.
    """
    user_id = runtime.context.user_id 

    if user_id in Company_db:
        user = Company_db[user_id]
        return f"User Name : {user['name']} and works in {user['department']}"
    return "User not found"

agent = create_agent(
    model = model,
    tools = [get_user_info],
    system_prompt= "You are receptionist at google."
)

In [22]:
response = agent.invoke(
    {
    "messages" : [("user","Which department do i belong ..?")]
    },
    context = UserContext(user_id="user1910")
)

print(response['messages'][-1].content)

You belong to the **Artificial Intelligence (AI)** department at Google, Dhairya! Let me know if you need any assistance. 😊


# Long Term Memory Store

In [32]:
from typing import Any 
from langchain.agents import create_agent
from langchain.tools import tool,ToolRuntime 
from langgraph.store.memory import InMemoryStore
from langgraph.checkpoint.memory import InMemorySaver

In [63]:
@dataclass 
class Userdata:
    user_id : str

@tool 
def get_user_details(user_id : str, runtime:ToolRuntime) -> str:
    """search for user's details"""
    user_detail = None 
    store = runtime.store 

    if runtime.context.user_id == user_id :
        user_detail = store.get(("users",),user_id)
    else :
        return "this user_id is not Authorized to access this details."
    
    return str(user_detail.value) if user_detail else "This user may not exist"

@tool 
def put_user_details(username:str, user_id:str, user_info:dict[str,Any], runtime:ToolRuntime) -> str:
    """
    Store used details.
    """
    store = runtime.store 
    store.put(("users",),user_id,user_info)
    return "User info saved successfully."

In [66]:
agent = create_agent(
    model = model,
    tools = [get_user_details,put_user_details],
    store = InMemoryStore(),
    checkpointer= InMemorySaver(),
    system_prompt="You are a receptionist.",
)

In [ ]:
config = {"configurable":{"thread_id":"1"}}
response = agent.invoke(
    {
        "messages" : [
            {
                "role":"user",
                "content":"retrive the information of user : user21" 
            }
            ],
    },
    config = config,
    context = UserContext(user_id="user22") # here user_id is user22 and the user is trying to access the info of other user which is not authorized. 
)

d:\AppstoneLab-AI-intern\Concept-Wise\Gen-AI\venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=UserContext(user_id='user22'), input_type=UserContext])
  return self.__pydantic_serializer__.to_python(


In [72]:
print(response['messages'][-1].content)

I'm sorry, but I am unable to retrieve the details for user **user21** as this request is not authorized. If you believe this is an error or need further assistance, please let me know!


In [73]:
response

{'messages': [HumanMessage(content='Store my details My name is Dhairya, user_id user21, department AI, email : ai@gmail.com', additional_kwargs={}, response_metadata={}, id='f31ddfe0-a07a-4d80-9408-b38fa700a38d'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'Ilo9ciHOk', 'function': {'name': 'put_user_details', 'arguments': '{"user_info": {"department": "AI", "email": "ai@gmail.com"}, "username": "Dhairya", "user_id": "user21"}'}, 'index': 0}]}, response_metadata={'token_usage': {'prompt_tokens': 184, 'total_tokens': 225, 'completion_tokens': 41, 'prompt_tokens_details': {'cached_tokens': 0}}, 'model_name': 'mistral-medium-2508', 'model': 'mistral-medium-2508', 'finish_reason': 'tool_calls', 'model_provider': 'mistralai'}, id='lc_run--019cdc57-731a-7451-b194-2a7ca2f3b47c-0', tool_calls=[{'name': 'put_user_details', 'args': {'user_info': {'department': 'AI', 'email': 'ai@gmail.com'}, 'username': 'Dhairya', 'user_id': 'user21'}, 'id': 'Ilo9ciHOk', 'type': 'tool_call'